# Physics of Nimbus

This section presents a summary of the physics included in Nimbus. More details can be found in
- [Kiefer et al. (2026)](https://arxiv.org/pdf/2603.13167) for the basics of Nimbus (v1.0).
- [Kiefer et al. (in prep)]() for mixed materials, coagulation, and gravitational coalescence. This version also includes external material input (v2.0).

## Master Equations

Nimbus calculates the Mass Mixing Ratios (MMRs) of cloud forming materials throughout the atmosphere. It considers gas-phase transport through diffusion, cloud particle transport through diffusion and gravitational settling, nucleation through MCNT, and accretion of material in the diffusion and collision limited regime. The master equations are:
$$\rho_a \frac{d q_v^\mathrm{mat}}{dt} = -\frac{\partial}{\partial z} K_{zz} \rho_a \frac{\partial q_v^\mathrm{mat}}{\partial z} - J^\mathrm{mat} m_\mathrm{n} - G^\mathrm{mat} m_\mathrm{1}^\mathrm{mat} + F_v^\mathrm{mat}  \tag{1}$$
$$\rho_a \frac{d q_c^\mathrm{mat}}{dt} = \frac{\partial}{\partial z} q_c^\mathrm{mat} \rho_a v_\mathrm{dr} - \frac{\partial}{\partial z} K_{zz} \rho_a \frac{\partial q_c^\mathrm{mat}}{\partial z} + J^\mathrm{mat} m_\mathrm{n} + G^\mathrm{mat} m_\mathrm{1}^\mathrm{mat} + F_c^\mathrm{mat} + F_n^\mathrm{mat} \tag{2}$$
$$\rho_a \frac{d q_n}{dt} = \frac{\partial}{\partial z} q_n \rho_a v_\mathrm{dr} - \frac{\partial}{\partial z} K_{zz} \rho_a \frac{\partial q_n}{\partial z} - m_n f_\mathrm{coag} - m_n f_\mathrm{coal} + \sum_\mathrm{mat} \left( J^\mathrm{mat} m_\mathrm{n} + F_n^\mathrm{mat} \right)  \tag{3}$$
where the variables are defined as:
<ul class="adaptive-list">
    <li>$q_v^\mathrm{mat}$ [g/g] is MMR of the gas-phase material,</li>
    <li>$q_c^\mathrm{mat}$ [g/g] is the MMR of the cloud particle material,</li>
    <li>$q_n$ [g/g] is the MMR of the Cloud Condensation Nucleai (CCN),</li>
    <li>$K_{zz}$ [cm$^2$/s] is the atmospheric mixing constant,</li>
    <li>$\rho_a$ [g/cm$^3$] is the atmospheric density,</li>
    <li>$z$ [cm] is the altitude,</li>
    <li>$t$ [s] is the time,</li>
    <li>$m_\mathrm{n}$ [g] is the CCN mass,</li>
    <li>$m_1^\mathrm{mat}$ [g] is the gas-phase mass of the cloud forming material,</li>
    <li>$J^\mathrm{mat}$ [1/cm$^{3}$/s] is the nucleation rate,</li>
    <li>$G^\mathrm{mat}$ [1/cm$^{3}$/s] is the accretion rate,</li>
    <li>$v_\mathrm{dr}$ [cm/s] is the settling velocity,</li>
    <li>$F_v^\mathrm{mat}$ [g/cm$^3$/s}] is the gas injection function,</li>
    <li>$F_c^\mathrm{mat}$ [g/cm$^3$/s}] is the cloud mass injection function,</li>
    <li>$F_n^\mathrm{mat}$ [g/cm$^3$/s}] is the CCN injection function,</li>
    <li>$f_\mathrm{coag}$ [1/cm$^3$/s] is the coagulation rate,</li>
    <li>$f_\mathrm{coag}$ [1/cm$^3$/s] is the coalessence rate,</li>
</ul>

Additional parameters needed to calculate the cloud particle physics are calculated during runtime. This includes: the cloud particle number density $n_c$ [1/cm$^3$], the total cloud particle density $\rho_c$ [g/cm$^3$], the average cloud particle mass $m_c$[g], and the cloud particle radius $r_c$ [cm]:
$$n_c = \frac{q_n \rho_a}{m_\mathrm{n}} \tag{4}$$
$$\rho_c = \frac{\sum_\mathrm{mat} q_c^\mathrm{mat}}{\sum q_c^\mathrm{mat}/\rho_c^\mathrm{mat}} \tag{5}$$
$$m_c = \frac{m_n }{q_n} \sum_\mathrm{mat} q_c^\mathrm{mat} \tag{6}$$
$$r_c = \sqrt[3]{\frac{3 m_c}{4 \pi \rho_c}} \tag{7}$$
Nimbus can account for multiple cloud particle species that form onto each other, resulting in heterogeneous cloud particles. For more information on the general physics of mixed particles, see [Helling & Woitke 2006](https://www.aanda.org/articles/aa/abs/2006/31/aa4598-05/aa4598-05.html). In Nimbus, Eq. 1 and 2 are solved for each cloud material seperatly. Certain species, for example MgSiO$_3$, are known to nucleate inefficiently. These species can be 'deactivated' as nucleators to better represent the heterogeneous growth of particles in exoplanet atmospheres.


<style>
.adaptive-list {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(450px, 1fr));
  gap: 0.1rem 1rem;
}
</style>

## Cloud Physics

In this section, we cover the microphysical processes included in Nimbus. To learn more how to modify each property within Nimbus check out the [Tutorials](Tutorials.ipynb).

### Settling Velocity

The gravitational acceleration of cloud particles is counteracted by the frictional force of the surrounding gas. The balance between these two forces will determine the terminal velocity $v_\mathrm{dr}$ [cm/s] of the cloud particles. Since acceleration timescales are typically short, it can be assumed that cloud particles settle with $v_\mathrm{dr}$. The strength of the frictional force depends on the cloud particle and atmospheric properties. There are two regimes which can be distinguished using the Knudsen number. If the Knudsen number is large ($Kn \gg 1$), the drag force can be described by a free molecular flow. This is called the Epstein regime. If the Knudsen number is small ($Kn \ll 1$), this is called the Stokes regime. For Nimbus, we use the interpolation scheme of [Huang et al. 2024](https://www.aanda.org/articles/aa/abs/2024/11/aa51112-24/aa51112-24.html):
$$v_\mathrm{dr} = \frac{g ~r ~\rho_c}{v_\mathrm{th}~\rho_a} \sqrt{1 + \left(\frac{4~r}{9~l} \right)^2} \tag{5}$$
where $g$ [cm/s$^2$] is the gravity, $v_\mathrm{th} = \sqrt{R_g T/(2\pi \mu_c)}$ [cm/s] is the thermal velocity, $R_g = 8.314 \times 10^7$ erg/(K~mol) is the ideal gas constant, $T$ [K] is the temperature of the atmosphere, and $\mu_c$ [amu] is the molecular weight of the cloud forming specie. This interpolation scheme results in a smooth transition between the high and low Knudsen number regimes.

### Nucleation Rate

Cloud formation in gas-giant planets starts with the formation of molecular clusters from the gas-phase. Modified Classical Nucleation Theory (MCNT) provieds an approximation of the nucleation rate in the absence of thermodynamic properties of larger clusters. The accuracy of this approximation depends on the nucleating species. The nucleation rate is given by:
$$J^\mathrm{mat} = 4 \pi r^2 ~ v_\mathrm{rel}^\mathrm{mat} ~Z ~ n_1^\mathrm{mat} ~n_{c} ~\exp(\Delta G^\mathrm{mat}/k_BT) s_\mathrm{nuc} \tag{6}$$
where $n_1^\mathrm{mat}$ [1/cm$^{3}$] is the number density of the nucleating species in the gas-phase, $v_\mathrm{rel}$ [cm/s] is the relative velocity, $Z$ is the Zeldovich factor, and $\Delta G$ [erg] the energy of formation. Since MCNT is used in cases where thermodynamic cluster data is missing, $\Delta G$ has to be derived through approximations. The variable $s_\mathrm{nuc}$ is a user input that allows to explor variations in the nucleation rate. It is by default set to 1.

### Accretion Rate

Once CCNs are present in the gas-phase, other materials can start to accrete. The rate at which materials accrete can either be collision or diffusion limited:
$$G^\mathrm{mat}_\mathrm{col} = 4 \pi r^2~ s~v_\mathrm{th}^\mathrm{mat} ~n_1^\mathrm{mat} ~n_{cl}  ~\left(1 - \frac{p_\mathrm{vap}^\mathrm{mat}}{p_1^\mathrm{mat}} \right) \tag{7}$$
$$G^\mathrm{mat}_\mathrm{dif} = 4 \pi r ~D^\mathrm{mat} ~n_1^\mathrm{mat} ~n_{cl} ~\left(1 - \frac{p_\mathrm{vap}^\mathrm{mat}}{p_1^\mathrm{mat}} \right) \tag{8}$$
where $G^\mathrm{mat}_\mathrm{col}$~[1/cm$^{3}$/s] is the growth rate in the collision limited regime, $G_\mathrm{dif}^\mathrm{mat}$ [1/cm$^{3}$/s] is the growth rate in the diffusion limited regime, $D^\mathrm{mat}$ [cm$^2$/s] is the gas phase diffusion constant. The variable $1\geq s >0$ denotes the sticking coefficient and can be used to fit cloud profiles to observations. For Nimbus, we use the tanh interpolation function for homogenous, mono-dispersed particles from [Lee (2023)](https://doi.org/10.1093/mnras/stad2037):
$$f_x = \frac{1}{2} \left( 1 - \tanh\left( 2 \log_{10}\left( \frac{G_\mathrm{dif}^\mathrm{mat}}{G_\mathrm{col}^\mathrm{mat}} \right) \right) \right) \tag{9}$$
$$G = G_\mathrm{dif}^\mathrm{mat} f_x + G_\mathrm{col}^\mathrm{mat} (1 - f_x) \tag{10}$$
This interpolation results in a smooth transition between both limits which is favourable for the numerical evaluation. Some cloud particle materials exist in the gas-phase and can condense directly onto cloud particles. This is true, for example, for SiO which can from through the following reaction:
$$\mathrm{SiO} \rightarrow \mathrm{SiO[s]}. \tag{11}$$
Other cloud particles will form through more complex surface reactions. For example, SiO$_2$ preferentially forms through:
$$\mathrm{SiO} + \mathrm{H_2O} \rightarrow \mathrm{SiO[s]} + \mathrm{H_2}. \tag{12}$$
The rate of these surface reactions, and therefore the vapour pressure, depends on the metallicity of the gas-phase.

### Coagulation and Gravitational Coalescence
The coagulation $f_\mathrm{coag}$ [Units of cm$^{-3}$s$^{-1}$] and gravitational coalescence rate $f_\mathrm{coal}$ [Units of cm$^{-3}$s$^{-1}$] are calculated following [Lee 2025](https://www.aanda.org/articles/aa/abs/2025/06/aa54511-25/aa54511-25.html) for mono-disperse particle distributions:
$$f_\mathrm{coag} = \frac{4 k_bT \beta}{3 \eta_a} n_c^2 \tag{13}$$
$$f_\mathrm{coal} = 2 \pi r_c^2 E\Delta v ~n_c^2 \tag{14}$$
\end{align}
where $\eta_a$ [Units of g~cm$^{-1}$s$^{-1}$] is the atmospheric dynamical viscosity, and $\Delta v$ [Units of cm~s$^{-1}$] is the relative velocity of cloud particles. The variables $\beta$ and $E$ are defined as:
$$\beta = 1 + \mathrm{Kn} \left[1.165 + 0.483 \exp \left(\frac{-0.997}{\mathrm{Kn}} \right) \right] \tag{15}$$
$$E = \begin{cases} \mathrm{Kn} < 1, & \max \left[ 0, 1 - 0.42 (\frac{v_\mathrm{dr} \Delta v}{g r_c})^{-0.75} \right] \\ \mathrm{Kn} \leq 1 & 1\end{cases} \tag{16}$$
where $\mathrm{Kn} = l/r_c$ is the Knudsen number, $l$ [Units of cm] is the mean free path, and $r_c$ [Units of cm] is the cloud particle radius.

### Adding New Species

 If you want to add a new cloud species add the necessary data to the nimbus/data/cloud_material.csv. If your species has a non-standard vapour pressure, you should mark the 'all data available' column as 'special' and add the vapour pressure function to the 'vapor_pressures' function in 'nimbus/src/species_database.py'. If you would like to contribute these new species to Nimbus, makes sure to add the references for your data.

## Numerics of Nimbus

Nimbus solves the ODEs using the 'LSODA' method of the solve_ivp function from SciPy which is based on DLSODES. All the standard settings are:
- use a relative tolerance of $r_\mathrm{tol} = 10^{-6}$
- use an absolute tolerance of $a_\mathrm{tol} = 10^{-25}$
- set all MMRs below $10^{-30}$ to 0
- assume a CCN radius of $r_\mathrm{CCN} = 10^{-3}$~$\mu$m

To achieve faster run times, the computational domain is limited to pressure layers where $S > 1$. To increase computational speed, the evaporation of cloud particle below the cloud layers ($S < 1$) is assumed to be instantaneous. To consider multiple cloud materials, Nimbus is run once for each material individually. The gas-phase chemistry affects cloud formation and vice versa. This causes feedback loops which lead to stiff ODEs. To prevent this, Nimbus includes two different approaches to calculate cloud structures:
- __full__ which solves the ODEs in their full complexity, or
- __iterative__ which assumes a fixed cloud particle radius that is re-calculated between iterations. The radius is smoothed using an 8th order polynomial to prevent numerical artifacts. Iterations are run until either a fixed number of iterations are reached (typically 10), or the maximum relative difference for all MMRs within all pressure layers is less than $10^{-3}$.

